In [1]:
import ee
import geemap
import os
from pathlib import Path 
os.chdir(r"/Users/hayashishingo/Documents/Dev/satellite/gee")
ee.Initialize(project="kyoto-gee-project")


In [2]:
# 夢洲・万博会場周辺
parking = ee.Geometry.Rectangle(
    [135.3900, 34.6450, 135.4200, 34.6650]
)
s1 = ee.ImageCollection('COPERNICUS/S1_GRD') \
  .filterBounds(parking) \
  .filterDate('2025-04-01', '2025-12-31') \
  .filter(ee.Filter.eq('instrumentMode', 'IW')) \
  .select('VV') \
  .sort('system:time_start')
  
# 確認
print('取得画像数:', s1.size().getInfo())

# 日付一覧
dates = s1.aggregate_array('system:time_start') \
    .map(lambda d: ee.Date(d).format('YYYY-MM-dd'))
print('取得日付:', dates.getInfo())



取得画像数: 40
取得日付: ['2025-04-05', '2025-04-11', '2025-04-23', '2025-05-05', '2025-05-11', '2025-05-17', '2025-05-23', '2025-05-29', '2025-06-04', '2025-06-10', '2025-06-16', '2025-06-22', '2025-06-28', '2025-07-04', '2025-07-16', '2025-07-22', '2025-07-28', '2025-08-03', '2025-08-09', '2025-08-15', '2025-08-21', '2025-08-27', '2025-09-02', '2025-09-14', '2025-09-20', '2025-09-26', '2025-10-02', '2025-10-08', '2025-10-20', '2025-10-26', '2025-11-01', '2025-11-07', '2025-11-13', '2025-11-19', '2025-11-25', '2025-12-01', '2025-12-07', '2025-12-13', '2025-12-19', '2025-12-25']


In [5]:

Map = geemap.Map()

# 日付と時間を取得
Map.add_basemap('SATELLITE',False)
Map.centerObject(parking, 14)
Map.addLayer(
    parking,
    {'color': 'red'},
    '夢洲・万博会場エリア'
)
for i in range(4):
    image = ee.Image(s1.toList(s1.size()).get(i))
    stats = image.reduceRegion(
        reducer=ee.Reducer.minMax(),
        geometry=parking,
        scale=10,
        maxPixels=1e9
    )
    v_max = stats.getInfo()["VV_max"]
    v_min = stats.getInfo()["VV_min"]
    
    datetime = image.date().format('YYYY-MM-dd HH:mm:ss').getInfo()
    Map.addLayer(
        image,
        {'min': v_min, 'max': v_max},
        f'{datetime}',
        False  # 最後にFalseで最初はOFF
    )
    print('撮影日時:', datetime)
Map

撮影日時: 2025-04-05 08:57:04
撮影日時: 2025-04-11 21:00:34
撮影日時: 2025-04-23 21:00:34
撮影日時: 2025-05-05 21:00:34


Map(center=[34.65500051625164, 135.40500000000003], controls=(WidgetControl(options=['position', 'transparent_…

In [ ]:
stats = image.reduceRegion(
    reducer=ee.Reducer.minMa
    geometry=parking,
    scale=10,
    maxPixels=1e9
)
print(stats.getInfo())

{'VV_max': 36.55650683930321, 'VV_min': -35.108569552034}


In [46]:
stats.getInfo()["VV_max"]

36.55650683930321